In [1]:
# 기본
import pandas as pd
import numpy as np

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split


# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
import pipe
import os
import gc

In [59]:
df=pipe.call_df(20)

In [61]:
df1=pd.read_csv('data/abnotab_pca3.csv',encoding='utf-8-sig')
df1=df1.drop('Segment',axis=1)
df2=pd.read_csv('data/C_AB_pca3.csv',encoding='utf-8-sig')


In [67]:
df2.columns=['C_pca1','C_pca2','C_pca3']

In [69]:
df=pd.concat([df,df1,df2],axis=1)

In [75]:
df=df.drop('ab',axis=1)

In [27]:
df.shape

(2400000, 26)

In [77]:

df1=df.copy()
# 입력과 결과로 나눈다.
X = df1.drop('Segment', axis=1)
print(X.columns,' 컬럼만 사용합니다')
y = df1['Segment']
# 문자열 -> 숫자

encoder1 = LabelEncoder()
encoder1.fit(y)
y2 = encoder1.transform(y)
# 입력 데이터 표준화
scaler1 = StandardScaler()
scaler1.fit(X)
X2 = scaler1.transform(X)
# 학습할 데이터를 변수에 담아준다.
# 학습용과 검증용으로 나눈다.
train_X, X_val, train_y, y_val = train_test_split(X2, y2, test_size=0.2, random_state=1)

Index(['_2순위카드이용금액', '이용건수_신용_R12M', '최대이용금액_일시불_R12M', '쇼핑_도소매_이용금액',
       '_1순위업종_이용금액', '_2순위업종_이용금액', '_3순위업종_이용금액', '_2순위쇼핑업종_이용금액',
       '_3순위쇼핑업종_이용금액', '_1순위교통업종_이용금액', '이용금액_오프라인_R6M', '이용건수_오프라인_R6M',
       '이용금액_오프라인_R3M', '이용금액_오프라인_B0M', '연체입금원금_B0M', '정상입금원금_B2M',
       '정상입금원금_B5M', '청구금액_B0', '잔액_일시불_B0M', '평잔_일시불_3M', 'PC1', 'PC2', 'PC3',
       'C_pca1', 'C_pca2', 'C_pca3'],
      dtype='object')  컬럼만 사용합니다


In [79]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score


C:\Users\katch\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [09:40:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)

KeyboardInterrupt



In [ ]:
param_grid = {
    'max_depth': [  6,7,8],
    'learning_rate': [ 0.2, 0.3, 0.4],
    'n_estimators': [200, 300]
}

grid = GridSearchCV(
    estimator=XGBClassifier(tree_method="hist", device="cuda",objective='multi:softmax', num_class=5, use_label_encoder=False, eval_metric='mlogloss'),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=3
)

grid.fit(train_X, train_y)

print("최적 F1:", grid.best_score_)
print("최적 파라미터:", grid.best_params_)

C:\Users\katch\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:07:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\katch\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:07:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\katch\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:07:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\katch\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [14:08:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtr

In [81]:
best_model = XGBClassifier()
#best_model.set_params(**grid.best_params_,num_class=5,tree_method="hist", device="cuda",eval_metric='mlogloss')
best_model.set_params(**{'learning_rate': 0.3, 'max_depth': 5, 'n_estimators': 300},num_class=5,tree_method="hist", device="cuda",eval_metric='mlogloss')
best_model.fit(train_X, train_y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.3, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None, num_class=5, ...)

In [85]:
from sklearn.metrics import classification_report
y_val_pred = best_model.predict(X_val)
print("최종 F1 (mi\cro):", f1_score(y_val, y_val_pred, average='macro'))
print(classification_report(y_val, y_val_pred, target_names=encoder1.classes_))

최종 F1 (macro): 0.5295264620118435
              precision    recall  f1-score   support

           A       0.74      0.34      0.47       203
           B       1.00      0.05      0.10        38
           C       0.67      0.51      0.58     25598
           D       0.63      0.51      0.56     69539
           E       0.91      0.96      0.94    384622

    accuracy                           0.87    480000
   macro avg       0.79      0.47      0.53    480000
weighted avg       0.86      0.87      0.86    480000



In [217]:
import pickle
with open('model/saved_model_26.dat', 'wb') as fp :
    pickle.dump(best_model, fp)
    pickle.dump(encoder1, fp)
    pickle.dump(scaler1, fp)


In [127]:
col_20=['_2순위카드이용금액', '쇼핑_도소매_이용금액', '_1순위교통업종_이용금액', '연체입금원금_B0M', '최대이용금액_일시불_R12M','잔액_일시불_B0M', '정상입금원금_B5M', '이용건수_신용_R12M', '_1순위업종_이용금액', '청구금액_B0', '_2순위업종_이용금액', '이용금액_오프라인_R6M','정상입금원금_B2M', '평잔_일시불_3M', '_3순위쇼핑업종_이용금액', '_3순위업종_이용금액','이용금액_오프라인_B0M', '이용건수_오프라인_R6M', '이용금액_오프라인_R3M', '_2순위쇼핑업종_이용금액']
col_ab=['카드이용한도금액_B1M'
,'카드이용한도금액_B2M'     
,'카드이용한도금액'       
,'CA한도금액'          
,'이용건수_선결제_R6M'    
,'선결제건수_R6M'        
,'상환개월수_결제일_R6M'   
,'이용금액_선결제_R6M'     
,'선입금원금_B2M'       
,'이용횟수_선결제_R6M'     
,'선입금원금_B0M'    ]
col_c=col=['청구금액_R6M'
,'CA한도금액'
,'할부금액_3M_R12M'
,'마일_적립포인트_R3M'
,'쇼핑_도소매_이용금액'
,'잔액_할부_무이자_B0M'
,'최대이용금액_일시불_R12M'
,'이용금액_오프라인_R3M'
,'_1순위업종_이용금액'
,'이용금액_해외']
col_all= list(set(col_20)| set(col_ab) |  set(col_c))

In [173]:
df_list=[]
path='./open/test'
for x in os.listdir(path):
    nxt_path=os.path.join(path,x)
   
    y = os.listdir(nxt_path)[-1] 
    file_path=os.path.join(nxt_path, y)    
    print(y)
    df=pd.read_parquet(file_path)
    df_list.append(df)
   
       
    
    
    gc.collect()   

merge_num=len(df_list)


  

if merge_num>1:   
    df=pd.merge(left=df_list[0],right=df_list[1],on=['ID','기준년월'])   
    if merge_num>2:
        for x in range(merge_num-2):
            df=pd.merge(left=df,right=df_list[x+2],on=['ID','기준년월'])   

df=df.drop(['ID','기준년월'],axis=1)

201812_test_회원정보.parquet
201812_test_신용정보.parquet
201812_test_승인매출정보.parquet
201812_test_청구정보.parquet
201812_test_잔액정보.parquet
201812_test_채널정보.parquet
201812_test_마케팅정보.parquet
201812_test_성과정보.parquet


In [177]:
df=df[col_all]

In [179]:
from sklearn.decomposition import PCA
x=df[col_ab]

# 스케일링
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

# PCA
pca = PCA(n_components=3)
X_pca = pca.fit_transform(x_scaled)
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2','PC3'])

In [181]:
x=df[col_c]

# 스케일링
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

# PCA
pca = PCA(n_components=3)
X_pca = pca.fit_transform(x_scaled)
df_pca_c = pd.DataFrame(X_pca, columns=['C_pca1', 'C_pca2', 'C_pca3'])

In [183]:
df_final=pd.concat([df[col_20],df_pca,df_pca_c],axis=1)

In [185]:
df_final=df_final[['_2순위카드이용금액', '이용건수_신용_R12M', '최대이용금액_일시불_R12M', '쇼핑_도소매_이용금액',
       '_1순위업종_이용금액', '_2순위업종_이용금액', '_3순위업종_이용금액', '_2순위쇼핑업종_이용금액',
       '_3순위쇼핑업종_이용금액', '_1순위교통업종_이용금액', '이용금액_오프라인_R6M', '이용건수_오프라인_R6M',
       '이용금액_오프라인_R3M', '이용금액_오프라인_B0M', '연체입금원금_B0M', '정상입금원금_B2M',
       '정상입금원금_B5M', '청구금액_B0', '잔액_일시불_B0M', '평잔_일시불_3M', 'PC1', 'PC2', 'PC3',
       'C_pca1', 'C_pca2', 'C_pca3']]
X=scaler1.transform(df_final)
result=encoder1.inverse_transform(best_model.predict(X))

In [209]:
submit_df = pd.DataFrame({
    'ID': [f'TEST_{x:05d}' for x in range(100000)],
    'Segment': result
})
submit_df.to_csv('data/submit26.csv',index=False)

#모델복원

In [ ]:
with open('model/saved_model_26.dat', 'rb') as fp :
    model1 = pickle.load(fp)
    encoder1 = pickle.load(fp)
    scaler1 = pickle.load(fp)

display(model1)
display(encoder1)
display(scaler1)

In [221]:
gc.collect()

13412